In [1]:
import sys
sys.path.append("..")

import numpy as np
from sklearn.dummy import DummyClassifier

from src.data import load_raw, clean, split_by_month, feature_frame, LABEL
from src.metrics import evaluate
from src.tracking import setup, log_run

df = clean(load_raw("../data/Base.csv"))
train, valid, test = split_by_month(df)   # the test set is never used in this notebook

In [2]:
perfect = evaluate(valid[LABEL], valid[LABEL].astype(float))
print({k: round(v, 4) for k, v in perfect.items()})

{'pr_auc': 1.0, 'recall_at_fpr_5pct': 1.0, 'recall_at_fpr_1pct': 1.0, 'recall_at_top_1pct': 0.8455, 'recall_at_top_5pct': 1.0, 'brier': 0.0}


In [3]:
setup()

X_train, y_train = feature_frame(train), train[LABEL]
X_valid, y_valid = feature_frame(valid), valid[LABEL]

model = DummyClassifier(strategy="prior").fit(X_train, y_train)
score = model.predict_proba(X_valid)[:, 1]
metrics = evaluate(y_valid, score)
print({k: round(v, 4) for k, v in metrics.items()})

log_run(
    "dummy_prior",
    params={
        "model": "DummyClassifier(prior)",
        "train_months": "0-4",
        "eval_split": "validation (month 5)",
        "n_features": X_train.shape[1],
        "days_since_request_included": False,
    },
    metrics={f"valid_{k}": v for k, v in metrics.items()},
)

2026/10/24 17:24:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/24 17:24:29 INFO mlflow.store.db.utils: Updating database tables


{'pr_auc': 0.0118, 'recall_at_fpr_5pct': 0.0454, 'recall_at_fpr_1pct': 0.0092, 'recall_at_top_1pct': 0.0092, 'recall_at_top_5pct': 0.0454, 'brier': 0.0117}
